<a href="https://colab.research.google.com/github/manojdm018/2050-college-landing-page/blob/main/ImageCompressionTool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 🖼️ IMAGE COMPRESSION WEB APP — Google Colab + Gradio + Pillow
# ============================================================
# STEP 1: Install required libraries (runs fine inside a Colab cell)
!pip install -q gradio pillow

# STEP 2: Import libraries
import gradio as gr          # For building the web UI
from PIL import Image        # For opening/compressing images
import io                    # For working with image data in memory
import os                    # For checking file sizes

# ------------------------------------------------------------
# HELPER FUNCTION 1: Convert raw bytes into a readable KB/MB string
# ------------------------------------------------------------
def format_size(size_bytes):
    if size_bytes < 1024 * 1024:
        return f"{size_bytes / 1024:.2f} KB"
    else:
        return f"{size_bytes / (1024 * 1024):.2f} MB"

# ------------------------------------------------------------
# HELPER FUNCTION 2: Show the original file size as soon as it's uploaded
# ------------------------------------------------------------
def show_original_info(filepath):
    if filepath is None:
        return ""
    size = os.path.getsize(filepath)
    return f"""
    <div style="padding:12px; background:#eef2ff; border-radius:10px;
                font-family:Arial; font-size:15px;">
        📁 <b>Original Size:</b> {format_size(size)}
    </div>
    """

# ------------------------------------------------------------
# MAIN FUNCTION: Compress the uploaded image and build the result UI
# ------------------------------------------------------------
def compress_and_show(filepath, quality, output_format):
    # Guard: no file uploaded yet
    if filepath is None:
        return None, "<p style='color:red;'>⚠️ Please upload an image first.</p>", gr.update(visible=False)

    # 1. Get the original file size directly from disk (most accurate)
    original_size = os.path.getsize(filepath)

    # 2. Open the image with Pillow
    img = Image.open(filepath)

    # JPEG doesn't support transparency, so convert RGBA/Palette images to RGB
    if output_format == "JPEG" and img.mode in ("RGBA", "P"):
        img = img.convert("RGB")

    # 3. Build the correct save settings depending on chosen format
    save_kwargs = {}
    if output_format == "JPEG":
        # Lower quality = smaller file, more compression artifacts
        save_kwargs = {"quality": quality, "optimize": True}
    elif output_format == "WEBP":
        # WEBP also supports a direct quality value (10-90)
        save_kwargs = {"quality": quality}
    elif output_format == "PNG":
        # PNG is LOSSLESS — "quality" doesn't reduce image detail,
        # so we map the slider to PNG's compression level (0-9) instead.
        # Lower slider value -> higher compression level -> smaller file.
        compress_level = max(0, min(9, int(9 - (quality / 90) * 9)))
        save_kwargs = {"optimize": True, "compress_level": compress_level}

    # 4. Save the compressed image into memory first (to measure its size)
    buffer = io.BytesIO()
    img.save(buffer, format=output_format, **save_kwargs)
    compressed_size = buffer.tell()

    # 5. Write the compressed image to disk so the user can download it
    ext = {"JPEG": "jpg", "PNG": "png", "WEBP": "webp"}[output_format]
    output_path = f"/content/compressed_image.{ext}"
    with open(output_path, "wb") as f:
        f.write(buffer.getvalue())

    # 6. Reload the saved file for the preview image
    compressed_preview_img = Image.open(output_path)

    # 7. Calculate the size reduction
    reduction_bytes = original_size - compressed_size
    reduction_percent = (reduction_bytes / original_size) * 100 if original_size > 0 else 0
    bar_width = max(0, min(100, reduction_percent))
    color = "#16a34a" if reduction_percent > 0 else "#dc2626"  # green if smaller, red if not

    # 8. Build a nicely formatted HTML summary (instead of raw text)
    stats_html = f"""
    <div style="font-family:Arial, sans-serif; padding:16px; border-radius:10px;
                background:#f8f9fa; border:1px solid #e0e0e0;">
        <table style="width:100%; font-size:15px; border-collapse:collapse;">
            <tr><td>📁 <b>Original Size</b></td>
                <td style="text-align:right;">{format_size(original_size)}</td></tr>
            <tr><td>📦 <b>Compressed Size</b></td>
                <td style="text-align:right;">{format_size(compressed_size)}</td></tr>
            <tr><td>💾 <b>Space Saved</b></td>
                <td style="text-align:right; color:{color}; font-weight:bold;">
                    {format_size(abs(reduction_bytes))} ({reduction_percent:.1f}%)
                </td></tr>
        </table>
        <div style="margin-top:12px; background:#e5e7eb; border-radius:8px;
                    overflow:hidden; height:22px;">
            <div style="width:{bar_width}%; background:{color}; height:100%;
                        text-align:center; color:white; font-size:12px; line-height:22px;
                        transition: width 0.4s;">
                {reduction_percent:.1f}% smaller
            </div>
        </div>
    </div>
    """

    return compressed_preview_img, stats_html, gr.update(value=output_path, visible=True)

# ------------------------------------------------------------
# BUILD THE GRADIO UI
# ------------------------------------------------------------
with gr.Blocks(title="Image Compressor") as demo:
    gr.Markdown("# 🖼️ Image Compression Tool")
    gr.Markdown("Upload an image, choose a quality level, and compress it instantly.")

    with gr.Row():
        # Left column: upload + original preview
        with gr.Column():
            gr.Markdown("### 1️⃣ Upload Image")
            input_image = gr.Image(type="filepath", label="Original Image", height=300)
            original_info = gr.HTML()  # shows original size right after upload

        # Right column: settings
        with gr.Column():
            gr.Markdown("### 2️⃣ Compression Settings")
            quality_slider = gr.Slider(
                minimum=10, maximum=90, value=50, step=5,
                label="Compression Quality (lower = smaller file, lower quality)"
            )
            format_dropdown = gr.Dropdown(
                choices=["JPEG", "PNG", "WEBP"], value="JPEG",
                label="Output Format"
            )
            compress_btn = gr.Button("🔄 Compress Image", variant="primary", size="lg")

    gr.Markdown("---")

    with gr.Row():
        # Left: compressed image preview
        with gr.Column():
            gr.Markdown("### 📤 Compressed Result")
            compressed_preview = gr.Image(label="Compressed Image", height=300, show_download_button=False)

        # Right: size comparison stats
        with gr.Column():
            gr.Markdown("### 📊 Size Comparison")
            stats_output = gr.HTML()

    download_btn = gr.DownloadButton(label="⬇️ Download Compressed Image", visible=False)

    # Event 1: as soon as a file is uploaded, show its original size
    input_image.change(
        fn=show_original_info,
        inputs=input_image,
        outputs=original_info
    )

    # Event 2: when "Compress Image" is clicked, run the compression
    compress_btn.click(
    fn=compress_and_show,
    inputs=[input_image, quality_slider, format_dropdown],
    outputs=[compressed_preview, stats_output, download_btn]  # <-- changed
)

# STEP 3: Launch the app (share=True gives you a public link from Colab)
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://89f8230f07d0f46f83.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 421, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 62, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1159, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error